In [1]:
import os
import time
import numpy as np
from scipy import ndimage

import torch
import torch.nn as nn
import torch.nn.parallel
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

from monai import transforms
from monai.networks.nets import SwinUNETR

# Importaciones de tu entorno
from src.get_data import UnifiedDataset
from src.custom_transforms import (
    ImputeMissingChannelsd,
    ConvertToMultiChannelPipeline2_ExperimentoD_d
)

# =======================================================
# 1. CONFIGURACIÓN Y TRANSFORMACIONES
# =======================================================
roi = (128, 128, 64) 
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Iniciando flujo Contrastivo en: {device}")

# Transformaciones para Extracción (Smart Crop)
extract_transform = transforms.Compose([
    transforms.LoadImaged(keys=["image", "label"]),
    transforms.EnsureChannelFirstd(keys=["image", "label"]),
    ImputeMissingChannelsd(keys=["image"]),
    ConvertToMultiChannelPipeline2_ExperimentoD_d(keys=["label"]),
    
    # El Francotirador: Busca las clases positivas (Infilt/Edema) y recorta ahí
    transforms.RandCropByPosNegLabeld(
        keys=["image", "label"],
        label_key="label",
        spatial_size=roi, # (128, 128, 64)
        pos=1,            # 100% de probabilidad de centrarse en la patología
        neg=0,            # 0% de probabilidad de caer en puro tejido sano
        num_samples=1,    # Extraemos 1 bloque brutalmente denso por paciente
        image_key="image",
        image_threshold=0,
    ),
    
    transforms.NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
])

# =======================================================
# 2. EXTRACCIÓN DE CARACTERÍSTICAS (HOOK)
# =======================================================
# Rutas
dataset_path = './Dataset/Dataset_30_6'
muglioma_path = './Dataset/MU_glioma'
# Usa el modelo de tu Experimento D 
model_path = "Dataset_Output/pipe2/solar-paper-11/model_best.pt" 

embedding_dir = "Dataset/contrastive_voxel_wise/exp_d/embeddings"
label_output_dir = "Dataset/contrastive_voxel_wise/exp_d/labels"
os.makedirs(embedding_dir, exist_ok=True)
os.makedirs(label_output_dir, exist_ok=True)

# Dataset unificado
train_set = UnifiedDataset(upenn_dir=dataset_path, muglioma_dir=muglioma_path, section="train", pipeline=2, transform=extract_transform)
train_loader = DataLoader(train_set, batch_size=1, shuffle=False, num_workers=2)




Iniciando flujo Contrastivo en: cuda:0
[TRAIN] Cargados 30 casos de UPenn-GBM (Pipeline 2)
[TRAIN] Cargados 190 casos de MU-Glioma Post (Pipeline 2)


In [ ]:
# Inicializar modelo SwinUNETR
model = SwinUNETR(img_size=roi, in_channels=11, out_channels=2, feature_size=48, use_checkpoint=True).to(device)

if os.path.exists(model_path):
    loaded_model = torch.load(model_path, map_location=device)["state_dict"]
    model.load_state_dict(loaded_model)
    print(f"Modelo base Exp D cargado desde {model_path}")
else:
    print(f"⚠️ Atención: No se encontró el modelo base. (Revisa la ruta)")

model.eval()

# Hook para el Decoder
decoder_features = None
def decoder_hook_fn(module, input, output):
    global decoder_features
    decoder_features = output

# Enganchamos en la última capa de decodificación antes de la salida final
hook_handle_decoder = model.decoder1.conv_block.register_forward_hook(decoder_hook_fn)

print("\n--- PASO 1: Extrayendo Vectores Latentes ---")
with torch.no_grad():
    for idx, batch_data in enumerate(train_loader):
        # Verifica si ya se extrajo
        if os.path.exists(f"{embedding_dir}/case_{idx}.npy"):
            continue
            
        # ==========================================
        # SOLUCIÓN AL TYPEERROR DE MONAI
        # ==========================================
        # Si MONAI devuelve una lista de recortes, sacamos el primer elemento
        if isinstance(batch_data, list):
            batch_data = batch_data[0]
            
        image = batch_data["image"].to(device)
        label = batch_data["label"].to(device) 
        
        # El DataLoader a veces anida el batch size sobre el num_samples [1, 1, C, H, W, D]
        # Si la imagen tiene 6 dimensiones, la aplanamos a 5D [Batch, C, H, W, D]
        if image.dim() == 6:
            image = image.squeeze(0)
            label = label.squeeze(0)
        # ==========================================
        
        label = label.squeeze(0) # [2, H, W, D]
        
        # --- MAPEO SEMÁNTICO EXPERIMENTO D ---
        # Canal 0: Infiltración Pura | Canal 1: Edema Vasogénico Puro
        label_class = torch.zeros_like(label[0], dtype=torch.long) # Todo Fondo (0)
        label_class[label[1] == 1] = 1 # 1 = Edema Vasogénico
        label_class[label[0] == 1] = 2 # 2 = Infiltración Pura
        
        label_np = label_class.cpu().numpy()
        
        # Forward pass (Activa el hook)
        with torch.cuda.amp.autocast():
            _ = model(image) 
        
        # Guardar como float16 para optimizar disco y VRAM
        np.save(f"{embedding_dir}/case_{idx}.npy", decoder_features.cpu().numpy().astype(np.float16))
        np.save(f"{label_output_dir}/case_{idx}.npy", label_np.astype(np.uint8))
        
        print(f"Caso {idx} guardado. Shape latente: {decoder_features.shape}")
        
        # Limpiar GPU
        del image, label, label_class, decoder_features
        torch.cuda.empty_cache()

hook_handle_decoder.remove()


In [3]:
# =======================================================
# 3. COMPONENTES DEL ENTRENAMIENTO CONTRASTIVO
# =======================================================

class EmbeddingDataset(Dataset):
    def __init__(self, embedding_dir, label_dir):
        self.embedding_dir = embedding_dir
        self.label_dir = label_dir
        self.case_files = [f for f in os.listdir(embedding_dir) if f.endswith(".npy")]
        
    def __len__(self):
        return len(self.case_files)
    
    def __getitem__(self, idx):
        file_name = self.case_files[idx]
        embedding_path = os.path.join(self.embedding_dir, file_name)
        label_path = os.path.join(self.label_dir, file_name)
        
        # Cargar y restaurar a float32
        embeddings = np.load(embedding_path).astype(np.float32)
        labels = np.load(label_path).astype(np.int64)
        
        return torch.tensor(embeddings).squeeze(0), torch.tensor(labels)

class ProjectionHead(nn.Module):
    """ Red neuronal profunda con LayerNorm para estabilizar el espacio latente """
    def __init__(self, input_dim=48, hidden_dim=256, output_dim=128):
        super(ProjectionHead, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, output_dim)
        )
    
    def forward(self, x):
        x = self.net(x)
        # Normalización L2 obligatoria para Supervised Contrastive Loss
        return F.normalize(x, dim=1)

def custom_sample_priority(embeddings_flat, labels_flat, max_infilt=10000, ratio_edema=2.0, ratio_bg=1.0):
    """
    Estrategia de Muestreo: SALVAR A LA MINORÍA.
    Toma toda la infiltración disponible (hasta max_infilt). 
    Luego, toma Edema y Fondo proporcionalmente para asegurar un lote balanceado.
    """
    idx_bg = (labels_flat == 0).nonzero(as_tuple=True)[0]
    idx_edema = (labels_flat == 1).nonzero(as_tuple=True)[0]
    idx_infilt = (labels_flat == 2).nonzero(as_tuple=True)[0]

    n_infilt = len(idx_infilt)
    n_edema = len(idx_edema)
    n_bg = len(idx_bg)

    # 1. PRIORIDAD ABSOLUTA: Infiltración
    target_infilt = min(n_infilt, max_infilt)
    
    if target_infilt == 0:
        return torch.tensor([]), torch.tensor([])

    # 2. PROPORCIONALIDAD: Edema y Fondo
    # Si hay poca infiltración (ej. 500 vóxeles), forzamos un piso mínimo (ej. 2000) 
    # para que la red no olvide las otras clases en ese batch.
    base_calc = max(target_infilt, 2000) 
    
    target_edema = min(n_edema, int(base_calc * ratio_edema))
    target_bg = min(n_bg, int(base_calc * ratio_bg))

    # Muestreo aleatorio
    def sample_idx(indices, target_size):
        if target_size <= 0 or len(indices) == 0: 
            return torch.tensor([], dtype=torch.long, device=indices.device)
        perm = torch.randperm(len(indices), device=indices.device)[:target_size]
        return indices[perm]

    s_infilt = sample_idx(idx_infilt, target_infilt)
    s_edema = sample_idx(idx_edema, target_edema)
    s_bg = sample_idx(idx_bg, target_bg)

    # Unir y mezclar
    all_idx = torch.cat([s_infilt, s_edema, s_bg])
    all_idx = all_idx[torch.randperm(len(all_idx))]

    return embeddings_flat[all_idx], labels_flat[all_idx]

def supcon_loss(features, labels, temperature=0.1):
    """
    Supervised Contrastive Loss con estabilidad numérica (max_sim trick).
    Junta las clases iguales, aleja las clases distintas.
    """
    device = features.device
    batch_size = features.shape[0]

    # Matriz de similitud y máscara de clases iguales
    similarity_matrix = torch.matmul(features, features.T) / temperature
    mask = (labels.unsqueeze(1) == labels.unsqueeze(0)).float().to(device)

    # Eliminar self-contrast (la diagonal no cuenta)
    logits_mask = torch.scatter(
        torch.ones_like(mask),
        1,
        torch.arange(batch_size).view(-1, 1).to(device),
        0
    )
    mask = mask * logits_mask

    # TRUCO DE ESTABILIDAD: Restar el máximo para evitar que torch.exp() de NaN
    sim_max, _ = torch.max(similarity_matrix, dim=1, keepdim=True)
    logits = similarity_matrix - sim_max.detach()

    # Denominador: Suma de todos los negativos + positivos (excepto uno mismo)
    exp_logits = torch.exp(logits) * logits_mask
    log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-9)

    # Numerador: Promedio log-prob de los positivos
    mask_sum = mask.sum(1)
    # Evitar división por cero si una clase solo tiene 1 ejemplo en el batch
    mask_sum[mask_sum == 0] = 1.0 
    
    mean_log_prob_pos = (mask * log_prob).sum(1) / mask_sum
    loss = -mean_log_prob_pos.mean()

    return loss

# =======================================================
# 4. CICLO DE ENTRENAMIENTO DE LA CABEZA DE PROYECCIÓN
# =======================================================
print("\n--- PASO 2: Entrenando Cabeza de Proyección Contrastiva ---")

dataset = EmbeddingDataset(embedding_dir, label_output_dir)
loader = DataLoader(dataset, batch_size=1, shuffle=True, num_workers=2)

proj_head = ProjectionHead(input_dim=48, hidden_dim=256, output_dim=128).to(device)
optimizer = optim.AdamW(proj_head.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

output_dir = "Dataset_Output/checkpoints_contrastive"
os.makedirs(output_dir, exist_ok=True)
best_model_path = os.path.join(output_dir, "best_contrastive_head_exp_d.pth")

num_epochs = 50
patience = 8
best_loss = float('inf')
epochs_no_improve = 0

for epoch in range(num_epochs):
    proj_head.train()
    total_loss = 0
    valid_batches = 0
    
    for batch_idx, (emb, lbl) in enumerate(loader):
        # 1. Eliminar la dimensión de batch que añade el DataLoader [1, 48, H, W, D] -> [48, H, W, D]
        emb = emb.to(device).squeeze(0)
        lbl = lbl.to(device).squeeze(0)
        
        # 2. Permutar para enviar las 48 características al final [H, W, D, 48]
        emb = emb.permute(1, 2, 3, 0) 
        
        # 3. Aplanar todo a 2D para el clasificador [N_voxeles, 48]
        emb_flat = emb.reshape(-1, 48)
        lbl_flat = lbl.reshape(-1)
        
        # Limpieza de valores nulos si los hay
        valid_mask = lbl_flat >= 0
        emb_valid = emb_flat[valid_mask]
        lbl_valid = lbl_flat[valid_mask]
        
        # MUESTREO (Prioridad Infiltración)
        emb_sampled, lbl_sampled = custom_sample_priority(
            emb_valid, lbl_valid, 
            max_infilt=10000, 
            ratio_edema=1.5, # Por cada voxel de Infilt, tomamos 1.5 de Edema
            ratio_bg=1.0     # Por cada voxel de Infilt, tomamos 1.0 de Fondo
        )
        
        if emb_sampled.shape[0] < 100:
            continue # Batch demasiado pequeño
            
        # Sub-batching para proteger la memoria VRAM durante la matriz NxN
        # 12000 x 12000 floats son ~576 MB. Es seguro.
        batch_size_clf = 10000 
        batch_loss_acum = 0
        sub_batches = 0
        
        for i in range(0, emb_sampled.shape[0], batch_size_clf):
            z_batch = emb_sampled[i:i+batch_size_clf]
            lbl_batch = lbl_sampled[i:i+batch_size_clf]
            
            # Forward
            z = proj_head(z_batch)
            loss = supcon_loss(z, lbl_batch, temperature=0.1) # T=0.1 obliga clusters más densos
            
            if torch.isnan(loss):
                continue
                
            # Backward
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            batch_loss_acum += loss.item()
            sub_batches += 1
            
        if sub_batches > 0:
            avg_batch_loss = batch_loss_acum / sub_batches
            total_loss += avg_batch_loss
            valid_batches += 1
            
            if batch_idx % 10 == 0:
                counts = np.bincount(lbl_sampled.cpu().numpy(), minlength=3)
                print(f"Ep {epoch+1} | Batch {batch_idx} | Loss: {avg_batch_loss:.4f} | "
                      f"Vóxeles [Fondo: {counts[0]}, Edema: {counts[1]}, Infilt: {counts[2]}]")

    if valid_batches == 0:
        continue
        
    avg_epoch_loss = total_loss / valid_batches
    scheduler.step(avg_epoch_loss)
    
    print(f"\n---> FIN ÉPOCA {epoch+1} | Loss Medio: {avg_epoch_loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}\n")
    
    if avg_epoch_loss < best_loss:
        best_loss = avg_epoch_loss
        epochs_no_improve = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': proj_head.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': best_loss,
        }, best_model_path)
    else:
        epochs_no_improve += 1
        
    if epochs_no_improve >= patience:
        print(f"Early Stopping! Mejor pérdida: {best_loss:.4f}")
        break

print("Entrenamiento Contrastivo Finalizado con Éxito.")


--- PASO 2: Entrenando Cabeza de Proyección Contrastiva ---


Ep 1 | Batch 0 | Loss: 8.8234 | Vóxeles [Fondo: 10000, Edema: 15000, Infilt: 10000]
Ep 1 | Batch 10 | Loss: 8.5925 | Vóxeles [Fondo: 2672, Edema: 4008, Infilt: 2672]
Ep 1 | Batch 20 | Loss: 7.9245 | Vóxeles [Fondo: 2000, Edema: 3000, Infilt: 561]
Ep 1 | Batch 30 | Loss: 8.7551 | Vóxeles [Fondo: 2783, Edema: 4174, Infilt: 2783]
Ep 1 | Batch 50 | Loss: 8.4360 | Vóxeles [Fondo: 10000, Edema: 15000, Infilt: 10000]
Ep 1 | Batch 60 | Loss: 8.6248 | Vóxeles [Fondo: 7695, Edema: 11542, Infilt: 7695]
Ep 1 | Batch 70 | Loss: 8.4889 | Vóxeles [Fondo: 10000, Edema: 15000, Infilt: 10000]
Ep 1 | Batch 80 | Loss: 8.0554 | Vóxeles [Fondo: 2000, Edema: 3000, Infilt: 2]
Ep 1 | Batch 90 | Loss: 8.1978 | Vóxeles [Fondo: 2000, Edema: 3000, Infilt: 1467]
Ep 1 | Batch 100 | Loss: 7.6366 | Vóxeles [Fondo: 3216, Edema: 4824, Infilt: 3216]
Ep 1 | Batch 110 | Loss: 8.6966 | Vóxeles [Fondo: 10000, Edema: 15000, Infilt: 10000]
Ep 1 | Batch 120 | Loss: 8.6480 | Vóxeles [Fondo: 10000, Edema: 15000, Infilt: 10000]
Ep

In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

# =======================================================
# 1. CONFIGURACIÓN BÁSICA
# =======================================================
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Iniciando Linear Probing en: {device}")

embedding_dir = "Dataset/contrastive_voxel_wise/exp_d/embeddings"
label_dir = "Dataset/contrastive_voxel_wise/exp_d/labels"

# Rutas de pesos
proj_head_weights = "Dataset_Output/checkpoints_contrastive/best_contrastive_head_exp_d.pth"
output_dir = "Dataset_Output/classifiers/checkpoints_mlp_probe"
os.makedirs(output_dir, exist_ok=True)
best_classifier_path = os.path.join(output_dir, "best_mlp_classifier_exp_d.pth")

# =======================================================
# 2. DATASET Y MUESTREO (BALANCE DE CLASES)
# =======================================================
class EmbeddingDataset(Dataset):
    def __init__(self, embedding_dir, label_dir):
        self.embedding_dir = embedding_dir
        self.label_dir = label_dir
        self.case_files = [f for f in os.listdir(embedding_dir) if f.endswith(".npy")]
        
    def __len__(self):
        return len(self.case_files)
    
    def __getitem__(self, idx):
        file_name = self.case_files[idx]
        emb_path = os.path.join(self.embedding_dir, file_name)
        lbl_path = os.path.join(self.label_dir, file_name)
        
        # Cargamos float16 y convertimos a float32 para la red
        embeddings = np.load(emb_path).astype(np.float32)
        labels = np.load(lbl_path).astype(np.int64)
        
        return torch.tensor(embeddings), torch.tensor(labels)

def custom_sample_priority(embeddings_flat, labels_flat, max_infilt=15000, ratio_edema=1.5, ratio_bg=1.0):
    """ Muestreo priorizando la infiltración para el clasificador lineal """
    idx_bg = (labels_flat == 0).nonzero(as_tuple=True)[0]
    idx_edema = (labels_flat == 1).nonzero(as_tuple=True)[0]
    idx_infilt = (labels_flat == 2).nonzero(as_tuple=True)[0]

    n_infilt, n_edema, n_bg = len(idx_infilt), len(idx_edema), len(idx_bg)
    target_infilt = min(n_infilt, max_infilt)
    
    if target_infilt == 0:
        return torch.tensor([]), torch.tensor([])

    base_calc = max(target_infilt, 2000) 
    target_edema = min(n_edema, int(base_calc * ratio_edema))
    target_bg = min(n_bg, int(base_calc * ratio_bg))

    def sample_idx(indices, target_size):
        if target_size <= 0 or len(indices) == 0: 
            return torch.tensor([], dtype=torch.long, device=indices.device)
        perm = torch.randperm(len(indices), device=indices.device)[:target_size]
        return indices[perm]

    s_infilt = sample_idx(idx_infilt, target_infilt)
    s_edema = sample_idx(idx_edema, target_edema)
    s_bg = sample_idx(idx_bg, target_bg)

    all_idx = torch.cat([s_infilt, s_edema, s_bg])
    all_idx = all_idx[torch.randperm(len(all_idx))]

    return embeddings_flat[all_idx], labels_flat[all_idx]

# =======================================================
# 3. DEFINICIÓN DE ARQUITECTURA
# =======================================================
class ProjectionHead(nn.Module):
    def __init__(self, input_dim=48, hidden_dim=256, output_dim=128):
        super(ProjectionHead, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, output_dim)
        )
    def forward(self, x):
        x = self.net(x)
        return F.normalize(x, dim=1)

# class LinearClassifier(nn.Module):
#     """ Una única capa lineal para clasificar: 0=Fondo, 1=Edema, 2=Infiltración """
#     def __init__(self, input_dim=128, num_classes=3):
#         super(LinearClassifier, self).__init__()
#         self.fc = nn.Linear(input_dim, num_classes)
        
#     def forward(self, x):
#         return self.fc(x)

# Tu Clasificador Supervisado (MLP)
class Classifier(nn.Module):
    def __init__(self, input_dim=128, hidden_dim1=256, hidden_dim2=128, num_classes=3, dropout_p=0.3):
        super(Classifier, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim1),
            nn.ReLU(),
            # En la extracción de características estáticas, el Dropout puede ayudar
            # a evitar el sobreajuste si la Infiltración tiene pocos vóxeles únicos.
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim2, num_classes)
        )
    
    def forward(self, x):
        return self.net(x)

# =======================================================
# 4. INICIALIZACIÓN
# =======================================================
dataset = EmbeddingDataset(embedding_dir, label_dir)
loader = DataLoader(dataset, batch_size=1, shuffle=True, num_workers=2)

# Instanciar y cargar Projection Head (CONGELADA)
proj_head = ProjectionHead().to(device)
if os.path.exists(proj_head_weights):
    checkpoint = torch.load(proj_head_weights, map_location=device)
    proj_head.load_state_dict(checkpoint['model_state_dict'])
    print(f"Pesos contrastivos cargados desde: {proj_head_weights}")
else:
    raise FileNotFoundError(f"No se encontró Projection Head en {proj_head_weights}")

proj_head.eval()
for param in proj_head.parameters():
    param.requires_grad = False # ¡Crucial para Linear Probing!

# Instanciar Clasificador (ENTRENABLE)
# classifier = LinearClassifier(input_dim=128, num_classes=3).to(device)

# Instanciar tu MLP Clasificador (ENTRENABLE)
classifier = Classifier(
    input_dim=128, 
    hidden_dim1=256, 
    hidden_dim2=128, 
    num_classes=3, 
    dropout_p=0.3
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(classifier.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=4)

# =======================================================
# 5. CICLO DE ENTRENAMIENTO SUPERVISADO
# =======================================================
print("\n--- PASO 3: Entrenando Clasificador Lineal ---")
num_epochs = 40
patience = 8
best_loss = float('inf')
epochs_no_improve = 0

for epoch in range(num_epochs):
    classifier.train()
    total_loss = 0
    valid_batches = 0
    
    # Para tracking de Accuracy
    correct_preds = {0: 0, 1: 0, 2: 0}
    total_preds = {0: 0, 1: 0, 2: 0}

    for batch_idx, (emb, lbl) in enumerate(loader):
        # 1. Destruimos TODAS las dimensiones de tamaño 1 sobrantes
        # emb pasa de [1, 1, 48, H, W, D] a [48, H, W, D]
        # lbl pasa de [1, 1, H, W, D] a [H, W, D]
        emb = emb.to(device).squeeze()
        lbl = lbl.to(device).squeeze()
        
        # 2. Ahora permutar es 100% seguro [H, W, D, 48]
        emb = emb.permute(1, 2, 3, 0) 
        
        # 3. Aplanamos para el MLP
        emb_flat = emb.reshape(-1, 48)
        lbl_flat = lbl.reshape(-1)
        
        valid_mask = lbl_flat >= 0
        emb_valid = emb_flat[valid_mask]
        lbl_valid = lbl_flat[valid_mask]
        
        emb_sampled, lbl_sampled = custom_sample_priority(emb_valid, lbl_valid)
        
        if emb_sampled.shape[0] < 100:
            continue
            
        batch_size_clf = 15000 
        batch_loss_acum = 0
        sub_batches = 0
        
        for i in range(0, emb_sampled.shape[0], batch_size_clf):
            z_batch = emb_sampled[i:i+batch_size_clf]
            labels_batch = lbl_sampled[i:i+batch_size_clf]
            
            # 1. Proyección (Sin gradientes)
            with torch.no_grad():
                contrastive_vectors = proj_head(z_batch)
                
            # 2. Clasificación (Con gradientes)
            logits = classifier(contrastive_vectors)
            loss = criterion(logits, labels_batch)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            batch_loss_acum += loss.item()
            sub_batches += 1
            
            # Tracking de métricas
            preds = torch.argmax(logits, dim=1)
            for c in range(3):
                class_mask = (labels_batch == c)
                correct_preds[c] += (preds[class_mask] == labels_batch[class_mask]).sum().item()
                total_preds[c] += class_mask.sum().item()

        if sub_batches > 0:
            total_loss += (batch_loss_acum / sub_batches)
            valid_batches += 1

    if valid_batches == 0:
        continue

    avg_epoch_loss = total_loss / valid_batches
    scheduler.step(avg_epoch_loss)
    
    # Calcular y mostrar Accuracy por clase
    acc_str = []
    class_names = {0: "Fondo", 1: "Edema", 2: "Infilt"}
    for c in range(3):
        acc = (correct_preds[c] / total_preds[c] * 100) if total_preds[c] > 0 else 0
        acc_str.append(f"{class_names[c]}: {acc:.1f}%")
        
    print(f"Ep {epoch+1:02d} | Loss: {avg_epoch_loss:.4f} | Acc -> {' | '.join(acc_str)}")
    
    if avg_epoch_loss < best_loss:
        best_loss = avg_epoch_loss
        epochs_no_improve = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': classifier.state_dict(),
            'loss': best_loss,
        }, best_classifier_path)
    else:
        epochs_no_improve += 1
        
    if epochs_no_improve >= patience:
        print(f"Early Stopping! Mejor pérdida: {best_loss:.4f}")
        break

print("Clasificador Lineal entrenado con éxito.")

Iniciando Linear Probing en: cuda:0
Pesos contrastivos cargados desde: Dataset_Output/checkpoints_contrastive/best_contrastive_head_exp_d.pth

--- PASO 3: Entrenando Clasificador Lineal ---
Ep 01 | Loss: 0.4316 | Acc -> Fondo: 93.8% | Edema: 88.5% | Infilt: 50.6%
Ep 02 | Loss: 0.3758 | Acc -> Fondo: 94.9% | Edema: 87.0% | Infilt: 59.2%
Ep 03 | Loss: 0.3740 | Acc -> Fondo: 95.0% | Edema: 87.8% | Infilt: 57.0%
Ep 04 | Loss: 0.3684 | Acc -> Fondo: 95.1% | Edema: 87.8% | Infilt: 56.7%
Ep 05 | Loss: 0.3686 | Acc -> Fondo: 95.4% | Edema: 88.1% | Infilt: 56.3%
Ep 06 | Loss: 0.3697 | Acc -> Fondo: 95.3% | Edema: 87.9% | Infilt: 57.1%
Ep 07 | Loss: 0.3647 | Acc -> Fondo: 95.3% | Edema: 88.3% | Infilt: 56.3%
Ep 08 | Loss: 0.3703 | Acc -> Fondo: 95.2% | Edema: 89.2% | Infilt: 54.2%
Ep 09 | Loss: 0.3642 | Acc -> Fondo: 95.3% | Edema: 88.6% | Infilt: 56.4%
Ep 10 | Loss: 0.3611 | Acc -> Fondo: 95.3% | Edema: 88.6% | Infilt: 56.0%
Ep 11 | Loss: 0.3642 | Acc -> Fondo: 95.2% | Edema: 89.1% | Infilt: 55

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from tqdm import tqdm

from monai.data import DataLoader
from monai.inferers import sliding_window_inference
from monai import transforms
from monai.networks.nets import SwinUNETR

# Importaciones de tu entorno (Asegúrate de que estas rutas sean correctas)
from src.get_data import UnifiedDataset
from src.custom_transforms import ImputeMissingChannelsd, ConvertToMultiChannelPipeline2_ExperimentoD_d

# ==========================================
# 1. CONFIGURACIÓN DE RUTAS Y MODELOS
# ==========================================
UPENN_DIR = "./Dataset/Dataset_30_6/"
MUGLIOMA_DIR = "./Dataset/MU_glioma/"
PARTICION = "val" 

# --- RUTAS DE LOS PESOS ---
MODEL_SWIN_WEIGHTS = "Dataset_Output/pipe2/solar-paper-11/model_best.pt" 
MODEL_PROJ_WEIGHTS = "Dataset_Output/checkpoints_contrastive/best_contrastive_head_exp_d.pth"
MODEL_MLP_WEIGHTS = "Dataset_Output/classifiers/checkpoints_mlp_probe/best_mlp_classifier_exp_d.pth"

roi = (128, 128, 64)
sw_batch_size = 2
infer_overlap = 0.5
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Iniciando evaluación Contrastiva en: {device}")

# ==========================================
# 2. DEFINICIÓN DE ARQUITECTURAS
# ==========================================
class ProjectionHead(nn.Module):
    def __init__(self, input_dim=48, hidden_dim=256, output_dim=128):
        super(ProjectionHead, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, output_dim)
        )
    def forward(self, x):
        return F.normalize(self.net(x), dim=1)

class Classifier(nn.Module):
    def __init__(self, input_dim=128, hidden_dim1=256, hidden_dim2=128, num_classes=3, dropout_p=0.3):
        super(Classifier, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim1),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim2, num_classes)
        )
    def forward(self, x):
        return self.net(x)

# --- EL MOTOR DE ENSAMBLAJE ---
class ContrastiveInferenceWrapper(nn.Module):
    """
    Toma la imagen 3D, pasa por SwinUNETR, roba las features internamente,
    las pasa por el Projection Head y el MLP, y devuelve los Logits 3D.
    """
    def __init__(self, swin, proj, clf):
        super().__init__()
        self.swin = swin
        self.proj = proj
        self.clf = clf
        self.decoder_features = None
        # Registramos el hook
        self.hook = self.swin.decoder1.conv_block.register_forward_hook(self.hook_fn)
        
    def hook_fn(self, module, input, output):
        self.decoder_features = output
        
    def forward(self, x):
        # 1. Pase por la red base (solo para activar el hook)
        _ = self.swin(x)
        feats = self.decoder_features # [B, 48, H, W, D]
        
        B, C, H, W, D = feats.shape
        
        # 2. Aplanar para las capas densas [N, 48]
        feats_flat = feats.permute(0, 2, 3, 4, 1).reshape(-1, C)
        
        # 3. Procesamiento en Sub-Lotes (Anti-OOM)
        # Procesamos 100,000 vóxeles a la vez para no saturar la RAM de la GPU
        chunk_size = 100000 
        logits_list = []
        
        for i in range(0, feats_flat.shape[0], chunk_size):
            f_chunk = feats_flat[i:i+chunk_size]
            p_chunk = self.proj(f_chunk)
            l_chunk = self.clf(p_chunk)
            logits_list.append(l_chunk)
            
        logits_flat = torch.cat(logits_list, dim=0) # [N, 3]
        
        # 4. Reconstrucción Espacial a [B, 3, H, W, D]
        logits = logits_flat.reshape(B, H, W, D, 3)
        logits = logits.permute(0, 4, 1, 2, 3)
        
        return logits

# ==========================================
# 3. CARGA DE MODELOS
# ==========================================
# SwinUNETR
swin_net = SwinUNETR(img_size=roi, in_channels=11, out_channels=2, feature_size=48, use_checkpoint=True).to(device)
swin_net.load_state_dict(torch.load(MODEL_SWIN_WEIGHTS, map_location=device)["state_dict"])

# Projection Head
proj_head = ProjectionHead().to(device)
proj_head.load_state_dict(torch.load(MODEL_PROJ_WEIGHTS, map_location=device)["model_state_dict"])

# Classifier (MLP)
mlp_clf = Classifier().to(device)
mlp_clf.load_state_dict(torch.load(MODEL_MLP_WEIGHTS, map_location=device)["model_state_dict"])

# Ensamblar Wrapper
model = ContrastiveInferenceWrapper(swin_net, proj_head, mlp_clf).to(device)
model.eval()

# ==========================================
# 4. PREPARACIÓN DE DATASET
# ==========================================
val_transform = transforms.Compose([
    transforms.LoadImaged(keys=["image", "label"]),
    transforms.EnsureChannelFirstd(keys=["image", "label"]),
    ImputeMissingChannelsd(keys=["image"]),
    ConvertToMultiChannelPipeline2_ExperimentoD_d(keys=["label"]),
    transforms.NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
])

dataset_eval = UnifiedDataset(upenn_dir=UPENN_DIR, muglioma_dir=MUGLIOMA_DIR, section=PARTICION, pipeline=2, transform=val_transform)
loader_eval = DataLoader(dataset_eval, batch_size=1, shuffle=False, num_workers=2)

# ==========================================
# 5. MOTOR DE EVALUACIÓN
# ==========================================
def calculate_dice(pred, target):
    smooth = 1e-5
    intersection = (pred * target).sum()
    return (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)

resultados_eval = []

with torch.no_grad():
    for batch_data in tqdm(loader_eval, desc="Evaluando SwinUNETR + Contrastive + MLP"):
        data = batch_data["image"].to(device)
        target = batch_data["label"].to(device) 
        source = batch_data.get("source", ["Desconocido"])[0]
        
        # Inferencia con Sliding Window (Llama internamente al wrapper y reconstruye)
        with torch.cuda.amp.autocast():
            logits = sliding_window_inference(data, roi, sw_batch_size, model, overlap=infer_overlap)
        
        # El modelo devuelve 3 canales de Logits. Usamos Argmax para obtener la clase dura
        # 0 = Fondo, 1 = Edema, 2 = Infiltración
        pred_classes = torch.argmax(logits[0], dim=0)
        
        # Separemos las máscaras booleanas
        pred_edema = (pred_classes == 1)
        pred_infilt = (pred_classes == 2)
        
        # Ground Truth (Del ConvertToMultiChannelPipeline2_ExperimentoD_d)
        # Canal 0: Infilt, Canal 1: Edema
        gt_infilt = target[0, 0].bool()
        gt_edema = target[0, 1].bool()
        
        # Cálculo de Dice
        dice_infilt = calculate_dice(pred_infilt, gt_infilt)
        dice_edema = calculate_dice(pred_edema, gt_edema)
        
        resultados_eval.append({
            "Dataset": source,
            "Dice_Infilt_Pura (Contrastivo)": dice_infilt.item(),
            "Dice_Edema_Puro (Contrastivo)": dice_edema.item()
        })
        
        del data, target, logits, pred_classes
        torch.cuda.empty_cache()

# ==========================================
# 6. REPORTE DE RESULTADOS
# ==========================================
df_eval = pd.DataFrame(resultados_eval)

print(f"\n{'='*70}")
print("REPORTE DE EVALUACIÓN: SwinUNETR + RED CONTRASTIVA + MLP")
print(f"{'='*70}")

columnas = ["Dice_Infilt_Pura (Contrastivo)", "Dice_Edema_Puro (Contrastivo)"]

print("\n--- MÉTRICAS GLOBALES ---")
print(df_eval[columnas].mean(numeric_only=True).to_frame("Dice Medio Global").round(4))

print("\n--- MÉTRICAS POR ORIGEN (UPENN vs MU-GLIOMA) ---")
print(df_eval.groupby("Dataset")[columnas].mean(numeric_only=True).round(4))

Iniciando evaluación Contrastiva en: cuda:0


/home/minigo/anaconda3/envs/monai_env/lib/python3.11/site-packages/monai/utils/deprecate_utils.py:221: FutureWarning: monai.networks.nets.swin_unetr SwinUNETR.__init__:img_size: Argument `img_size` has been deprecated since version 1.3. It will be removed in version 1.5. The img_size argument is not required anymore and checks on the input size are run during forward().
  warn_deprecated(argname, msg, warning_category)


[VAL] Cargados 6 casos de UPenn-GBM (Pipeline 2)
[VAL] Cargados 21 casos de MU-Glioma Post (Pipeline 2)


Evaluando SwinUNETR + Contrastive + MLP: 100%|██████████| 27/27 [01:48<00:00,  4.02s/it]


REPORTE DE EVALUACIÓN: SwinUNETR + RED CONTRASTIVA + MLP

--- MÉTRICAS GLOBALES ---
                                Dice Medio Global
Dice_Infilt_Pura (Contrastivo)             0.1493
Dice_Edema_Puro (Contrastivo)              0.6915

--- MÉTRICAS POR ORIGEN (UPENN vs MU-GLIOMA) ---
          Dice_Infilt_Pura (Contrastivo)  Dice_Edema_Puro (Contrastivo)
Dataset                                                                
MUGLIOMA                          0.1156                         0.6949
UPENN                             0.2672                         0.6795


## Visualizar mapas SwinUNETR + contrastive + clasiffier

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

from monai.inferers import sliding_window_inference
from monai import transforms
from monai.networks.nets import SwinUNETR

# Importar tus clases de datos unificadas
from src.get_data import UnifiedDataset
from src.custom_transforms import ImputeMissingChannelsd, ConvertToMultiChannelPipeline2_ExperimentoD_d

# Comando mágico para Jupyter
%matplotlib inline

# ==========================================
# 1. PARÁMETROS Y RUTAS DEL PIPELINE CONTRASTIVO
# ==========================================
UPENN_DIR = "./Dataset/Dataset_30_6/"
MUGLIOMA_DIR = "./Dataset/MU_glioma/"
roi = (128, 128, 64)
ID_BUSCADO = "UPENN-GBM-00055" 

MODEL_SWIN_WEIGHTS = "Dataset_Output/pipe2/solar-paper-11/model_best.pt" 
MODEL_PROJ_WEIGHTS = "Dataset_Output/checkpoints_contrastive/best_contrastive_head_exp_d.pth"
MODEL_MLP_WEIGHTS = "Dataset_Output/classifiers/checkpoints_mlp_probe/best_mlp_classifier_exp_d.pth"

# ==========================================
# 2. DEFINICIÓN DE REDES Y WRAPPER DE INFERENCIA
# ==========================================
class ProjectionHead(nn.Module):
    def __init__(self, input_dim=48, hidden_dim=256, output_dim=128):
        super(ProjectionHead, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, output_dim)
        )
    def forward(self, x):
        return F.normalize(self.net(x), dim=1)

class Classifier(nn.Module):
    def __init__(self, input_dim=128, hidden_dim1=256, hidden_dim2=128, num_classes=3, dropout_p=0.3):
        super(Classifier, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim1),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim2, num_classes)
        )
    def forward(self, x):
        return self.net(x)

class ContrastiveInferenceWrapper(nn.Module):
    def __init__(self, swin, proj, clf):
        super().__init__()
        self.swin = swin
        self.proj = proj
        self.clf = clf
        self.decoder_features = None
        self.hook = self.swin.decoder1.conv_block.register_forward_hook(self.hook_fn)
        
    def hook_fn(self, module, input, output):
        self.decoder_features = output
        
    def forward(self, x):
        _ = self.swin(x)
        feats = self.decoder_features 
        B, C, H, W, D = feats.shape
        feats_flat = feats.permute(0, 2, 3, 4, 1).reshape(-1, C)
        
        chunk_size = 100000 
        logits_list = []
        for i in range(0, feats_flat.shape[0], chunk_size):
            f_chunk = feats_flat[i:i+chunk_size]
            p_chunk = self.proj(f_chunk)
            l_chunk = self.clf(p_chunk)
            logits_list.append(l_chunk)
            
        logits_flat = torch.cat(logits_list, dim=0)
        logits = logits_flat.reshape(B, H, W, D, 3).permute(0, 4, 1, 2, 3)
        return logits

# ==========================================
# 3. CARGA E INFERENCIA VOLUMÉTRICA
# ==========================================
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
val_transform = transforms.Compose([
    transforms.LoadImaged(keys=["image", "label"]),
    transforms.EnsureChannelFirstd(keys=["image", "label"]),
    ImputeMissingChannelsd(keys=["image"]),
    ConvertToMultiChannelPipeline2_ExperimentoD_d(keys=["label"]),
    transforms.NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
])

test_set = UnifiedDataset(upenn_dir=UPENN_DIR, muglioma_dir=MUGLIOMA_DIR, section="val", pipeline=2, transform=val_transform)
indice_encontrado = next((i for i, d in enumerate(test_set.data) if ID_BUSCADO in d["label"]), None)

if indice_encontrado is not None:
    datos = test_set[indice_encontrado]
    imagen = datos["image"].unsqueeze(0).to(device)
    etiqueta = datos["label"].unsqueeze(0).to(device)
    
    # Cargar pesos y ensamblar
    swin_net = SwinUNETR(img_size=roi, in_channels=11, out_channels=2, feature_size=48, use_checkpoint=True).to(device)
    swin_net.load_state_dict(torch.load(MODEL_SWIN_WEIGHTS, map_location=device)["state_dict"])
    
    proj_head = ProjectionHead().to(device)
    proj_head.load_state_dict(torch.load(MODEL_PROJ_WEIGHTS, map_location=device)["model_state_dict"])
    
    mlp_clf = Classifier().to(device)
    mlp_clf.load_state_dict(torch.load(MODEL_MLP_WEIGHTS, map_location=device)["model_state_dict"])
    
    model_completo = ContrastiveInferenceWrapper(swin_net, proj_head, mlp_clf).to(device)
    model_completo.eval()
    
    print("Ejecutando inferencia completa con Sliding Window...")
    with torch.no_grad():
        with torch.cuda.amp.autocast():
            logits = sliding_window_inference(imagen, roi, sw_batch_size=2, predictor=model_completo, overlap=0.5)
            # Aplicamos Softmax multiclase [0: Fondo, 1: Edema, 2: Infiltración]
            probabilidades = F.softmax(logits, dim=1)[0].cpu().numpy()
            
    img_np = imagen[0].cpu().numpy()
    gt_np = etiqueta[0].cpu().numpy()
    
    max_z = img_np.shape[3] - 1
    corte_z_optimo = int(np.argmax(np.sum(gt_np[0], axis=(0, 1))))
    
    # ==========================================
    # 4. INTERFAZ DE VISUALIZACIÓN
    # ==========================================
    # OPTIMIZACIÓN: Calculamos la máscara dura de todo el volumen una sola vez
    pred_classes_volumen = np.argmax(probabilidades, axis=0) # Shape: [H, W, D]
    
    def plot_contrastive_analysis(corte_z):
        fig, axes = plt.subplots(2, 3, figsize=(18, 11))
        fig.suptitle(f'Análisis de Fallo Pipieline Contrastivo+MLP | Caso: {ID_BUSCADO} | Corte Z: {corte_z}', fontsize=16, y=0.98)
        
        # Secuencia FLAIR como fondo anatómico estándar para la periferia (índice 7)
        flair_bg = img_np[7, :, :, corte_z]
        
        # --- FILA 1: INFILTRACIÓN PURA (Canal 2 del MLP vs Canal 0 del GT) ---
        axes[0, 0].imshow(flair_bg, cmap="gray")
        mask_gt_infilt = np.ma.masked_where(gt_np[0, :, :, corte_z] == 0, gt_np[0, :, :, corte_z])
        axes[0, 0].imshow(mask_gt_infilt, cmap="Greens", alpha=0.6)
        axes[0, 0].set_title('Ground Truth: Infiltración', fontsize=12); axes[0, 0].axis('off')
        
        axes[0, 1].imshow(flair_bg, cmap="gray")
        # Mostramos la probabilidad cruda de infiltración (Canal 2) sin umbralizar
        p_infilt = probabilidades[2, :, :, corte_z]
        im0 = axes[0, 1].imshow(p_infilt, cmap="jet", alpha=0.6, vmin=0, vmax=1)
        axes[0, 1].set_title('Mapa de Probabilidad: Infiltración', fontsize=12); axes[0, 1].axis('off')
        fig.colorbar(im0, ax=axes[0, 1], fraction=0.046, pad=0.04)
        
        axes[0, 2].imshow(flair_bg, cmap="gray")
        # CORRECCIÓN: Extraemos el corte Z de la máscara dura calculada previamente
        pred_infilt_corte = (pred_classes_volumen[:, :, corte_z] == 2)
        mask_pred_infilt = np.ma.masked_where(~pred_infilt_corte, pred_infilt_corte)
        axes[0, 2].imshow(mask_pred_infilt, cmap="Oranges", alpha=0.7)
        axes[0, 2].set_title('Máscara Final (Argmax == 2)', fontsize=12); axes[0, 2].axis('off')
        
        # --- FILA 2: EDEMA VASOGÉNICO (Canal 1 del MLP vs Canal 1 del GT) ---
        axes[1, 0].imshow(flair_bg, cmap="gray")
        mask_gt_edema = np.ma.masked_where(gt_np[1, :, :, corte_z] == 0, gt_np[1, :, :, corte_z])
        axes[1, 0].imshow(mask_gt_edema, cmap="Blues", alpha=0.6)
        axes[1, 0].set_title('Ground Truth: Edema Puro', fontsize=12); axes[1, 0].axis('off')
        
        axes[1, 1].imshow(flair_bg, cmap="gray")
        p_edema = probabilidades[1, :, :, corte_z]
        im1 = axes[1, 1].imshow(p_edema, cmap="jet", alpha=0.6, vmin=0, vmax=1)
        axes[1, 1].set_title('Mapa de Probabilidad: Edema', fontsize=12); axes[1, 1].axis('off')
        fig.colorbar(im1, ax=axes[1, 1], fraction=0.046, pad=0.04)
        
        axes[1, 2].imshow(flair_bg, cmap="gray")
        # CORRECCIÓN: Extraemos el corte Z de la máscara dura calculada previamente
        pred_edema_corte = (pred_classes_volumen[:, :, corte_z] == 1)
        mask_pred_edema = np.ma.masked_where(~pred_edema_corte, pred_edema_corte)
        axes[1, 2].imshow(mask_pred_edema, cmap="Purples", alpha=0.7)
        axes[1, 2].set_title('Máscara Final (Argmax == 1)', fontsize=12); axes[1, 2].axis('off')
        
        plt.tight_layout()
        plt.show()

    slider = widgets.IntSlider(min=0, max=max_z, step=1, value=corte_z_optimo, description='Corte Z:')
    widgets.interact(plot_contrastive_analysis, corte_z=slider)
    model_completo.hook.remove()
else:
    print(f"ID {ID_BUSCADO} no localizado.")

[VAL] Cargados 6 casos de UPenn-GBM (Pipeline 2)
[VAL] Cargados 21 casos de MU-Glioma Post (Pipeline 2)


/home/minigo/anaconda3/envs/monai_env/lib/python3.11/site-packages/monai/utils/deprecate_utils.py:221: FutureWarning: monai.networks.nets.swin_unetr SwinUNETR.__init__:img_size: Argument `img_size` has been deprecated since version 1.3. It will be removed in version 1.5. The img_size argument is not required anymore and checks on the input size are run during forward().
  warn_deprecated(argname, msg, warning_category)


Ejecutando inferencia completa con Sliding Window...


interactive(children=(IntSlider(value=90, description='Corte Z:', max=154), Output()), _dom_classes=('widget-i…

## Visualizar mapas P1 + P2 Exp D

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap # <-- NUEVO: Importación para colores sólidos
import ipywidgets as widgets
from IPython.display import display

from monai.inferers import sliding_window_inference
from monai import transforms
from monai.networks.nets import SwinUNETR

# Importar tus utilidades
from src.get_data import UnifiedDataset
from src.custom_transforms import ImputeMissingChannelsd, ConvertToMultiChannel3Tier_d

# Comando mágico de Jupyter
%matplotlib inline

# ==========================================
# 1. SETUP DE RUTAS Y MODELOS (CASCADA)
# ==========================================
UPENN_DIR = "./Dataset/Dataset_30_6/"
MUGLIOMA_DIR = "./Dataset/MU_glioma/"
ID_BUSCADO = "UPENN-GBM-00285"
# ID_BUSCADO = "PatientID_0038"

# --- RUTAS DE LOS PESOS ---
MODEL_P1_WEIGHTS = "./Dataset_Output/pipe1/colorful-cloud-11/model_best.pt"
MODEL_P2D_WEIGHTS = "./Dataset_Output/pipe2/resilient-glade-12/model_best.pt"

roi = (128, 128, 64)
sw_batch_size = 2
infer_overlap = 0.5
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

val_transform = transforms.Compose([
    transforms.LoadImaged(keys=["image", "label"]),
    transforms.EnsureChannelFirstd(keys=["image", "label"]),
    ImputeMissingChannelsd(keys=["image"]),
    ConvertToMultiChannel3Tier_d(keys=["label"]), # 3-Tier para obtener el GT completo
    transforms.NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
])

test_set = UnifiedDataset(upenn_dir=UPENN_DIR, muglioma_dir=MUGLIOMA_DIR, section="val", pipeline=2, transform=val_transform)

print("Cargando Modelos en Cascada...")
model_p1 = SwinUNETR(img_size=roi, in_channels=11, out_channels=2, feature_size=48, use_checkpoint=True).to(device)
model_p1.load_state_dict(torch.load(MODEL_P1_WEIGHTS, map_location=device)["state_dict"])
model_p1.eval()

model_p2d = SwinUNETR(img_size=roi, in_channels=11, out_channels=2, feature_size=48, use_checkpoint=True).to(device)
model_p2d.load_state_dict(torch.load(MODEL_P2D_WEIGHTS, map_location=device)["state_dict"])
model_p2d.eval()

# ==========================================
# 2. INFERENCIA VOLUMÉTRICA Y MATEMÁTICA DE CASCADA
# ==========================================
indice_encontrado = next((i for i, d in enumerate(test_set.data) if ID_BUSCADO in d["label"]), None)

if indice_encontrado is not None:
    datos = test_set[indice_encontrado]
    imagen = datos["image"].unsqueeze(0).to(device) 
    etiqueta = datos["label"].unsqueeze(0) 
    
    with torch.no_grad():
        with torch.cuda.amp.autocast():
            # Inferencia P1
            logits_p1 = sliding_window_inference(imagen, roi, sw_batch_size, model_p1, overlap=infer_overlap)
            prob_p1 = torch.sigmoid(logits_p1)[0, 0].cpu().numpy() # [H, W, D] (Solo canal Core)
            
            # Inferencia P2 (Infilt vs Edema)
            logits_p2d = sliding_window_inference(imagen, roi, sw_batch_size, model_p2d, overlap=infer_overlap)
            prob_p2d = torch.sigmoid(logits_p2d)[0].cpu().numpy()

    img_np = imagen[0].cpu().numpy()
    gt_np = etiqueta[0].numpy()
    
    # --- OPTIMIZACIÓN: CÁLCULOS BOOLEANOS PREVIOS ---
    # 1. Ground Truth Diseccionado
    gt_core = gt_np[0]
    gt_infilt = gt_np[1] > gt_np[0]
    gt_edema = gt_np[2] > gt_np[1]
    
    # 2. Mapas de Probabilidad Crudos
    prob_infilt_cruda = prob_p2d[0]
    prob_edema = prob_p2d[1]
    
    # ==============================================================
    # 3. EL BISTURÍ JERÁRQUICO (Máscaras Mutuamente Excluyentes)
    # ==============================================================
    pred_core_dura = prob_p1 > 0.5
    prob_infilt_cascada = prob_infilt_cruda * (~pred_core_dura)
    prob_edema_cascada = prob_edema * (~pred_core_dura)
    
    # ==============================================================
    # 4. Máscaras Finales Duras (Para la 3ra columna)
    # ==============================================================
    pred_infilt_dura = (prob_infilt_cascada > 0.5) & (prob_infilt_cascada > prob_edema_cascada)
    pred_edema_dura = (prob_edema_cascada > 0.5) & (prob_edema_cascada >= prob_infilt_cascada)
    
    max_z = img_np.shape[3] - 1
    corte_z_optimo = int(np.argmax(np.sum(gt_infilt, axis=(0, 1))))
    
    # --- NUEVO: Definición de Colores Sólidos ---
    cmap_core = ListedColormap(['red'])
    cmap_edema = ListedColormap(['dodgerblue']) # dodgerblue resalta mejor que el azul oscuro
    cmap_infilt = ListedColormap(['lime'])      # verde lima para contrastar fuertemente
    
    # ==========================================
    # 3. MOTOR DE VISUALIZACIÓN INTERACTIVA
    # ==========================================
    def plot_campeon_analysis(corte_z):
        fig, axes = plt.subplots(3, 3, figsize=(18, 16))
        fig.suptitle(f'Arquitectura Campeona (Cascada Topológica P1+P2) | Caso: {ID_BUSCADO} | Z: {corte_z}', fontsize=18, y=0.98)

        t1gd_bg = img_np[9, :, :, corte_z]
        flair_bg = img_np[7, :, :, corte_z]
        
        # --------------------------------------------------
        # FILA 1: TUMOR CORE (Pipeline 1)
        # --------------------------------------------------
        axes[0, 0].imshow(t1gd_bg, cmap="gray")
        mask_gt_core = np.ma.masked_where(gt_core[:, :, corte_z] == 0, gt_core[:, :, corte_z])
        axes[0, 0].imshow(mask_gt_core, cmap=cmap_core, alpha=0.6) # Aplicando color sólido
        axes[0, 0].set_title('Ground Truth: Tumor Core', fontsize=14); axes[0, 0].axis('off')

        axes[0, 1].imshow(t1gd_bg, cmap="gray")
        p_core_map = prob_p1[:, :, corte_z]
        im_core = axes[0, 1].imshow(p_core_map, cmap="hot", alpha=0.6, vmin=0, vmax=1)
        axes[0, 1].set_title('Mapa de Probabilidad: P1 (Core)', fontsize=14); axes[0, 1].axis('off')
        fig.colorbar(im_core, ax=axes[0, 1], fraction=0.046, pad=0.04)

        axes[0, 2].imshow(t1gd_bg, cmap="gray")
        mask_pred_core = np.ma.masked_where(~pred_core_dura[:, :, corte_z], pred_core_dura[:, :, corte_z])
        axes[0, 2].imshow(mask_pred_core, cmap=cmap_core, alpha=0.8) # Aplicando color sólido
        axes[0, 2].set_title('Máscara Final (Umbral > 0.5)', fontsize=14); axes[0, 2].axis('off')

        # --------------------------------------------------
        # FILA 2: EDEMA VASOGÉNICO (Pipeline 2)
        # --------------------------------------------------
        axes[1, 0].imshow(flair_bg, cmap="gray")
        mask_gt_edema = np.ma.masked_where(gt_edema[:, :, corte_z] == 0, gt_edema[:, :, corte_z])
        axes[1, 0].imshow(mask_gt_edema, cmap=cmap_edema, alpha=0.6) # Aplicando color sólido
        axes[1, 0].set_title('Ground Truth: Edema Puro', fontsize=14); axes[1, 0].axis('off')

        axes[1, 1].imshow(flair_bg, cmap="gray")
        p_edema_map = prob_edema[:, :, corte_z]
        im_edema = axes[1, 1].imshow(p_edema_map, cmap="winter", alpha=0.6, vmin=0, vmax=1)
        axes[1, 1].set_title('Mapa de Probabilidad: P2 (Edema)', fontsize=14); axes[1, 1].axis('off')
        fig.colorbar(im_edema, ax=axes[1, 1], fraction=0.046, pad=0.04)

        axes[1, 2].imshow(flair_bg, cmap="gray")
        mask_pred_edema = np.ma.masked_where(~pred_edema_dura[:, :, corte_z], pred_edema_dura[:, :, corte_z])
        axes[1, 2].imshow(mask_pred_edema, cmap=cmap_edema, alpha=0.8) # Aplicando color sólido
        axes[1, 2].set_title('Máscara Final (Umbral > 0.5)', fontsize=14); axes[1, 2].axis('off')

        # --------------------------------------------------
        # FILA 3: INFILTRACIÓN (Operación en Cascada)
        # --------------------------------------------------
        axes[2, 0].imshow(flair_bg, cmap="gray")
        mask_gt_infilt = np.ma.masked_where(gt_infilt[:, :, corte_z] == 0, gt_infilt[:, :, corte_z])
        axes[2, 0].imshow(mask_gt_infilt, cmap=cmap_infilt, alpha=0.6) # Aplicando color sólido
        axes[2, 0].set_title('Ground Truth: Infiltración Pura', fontsize=14); axes[2, 0].axis('off')

        axes[2, 1].imshow(flair_bg, cmap="gray")
        p_infilt_map = prob_infilt_cascada[:, :, corte_z]
        im_infilt = axes[2, 1].imshow(p_infilt_map, cmap="summer", alpha=0.6, vmin=0, vmax=1)
        axes[2, 1].set_title('Prob. Cascada (P2 - P1 Core)', fontsize=14); axes[2, 1].axis('off')
        fig.colorbar(im_infilt, ax=axes[2, 1], fraction=0.046, pad=0.04)

        axes[2, 2].imshow(flair_bg, cmap="gray")
        mask_pred_infilt = np.ma.masked_where(~pred_infilt_dura[:, :, corte_z], pred_infilt_dura[:, :, corte_z])
        axes[2, 2].imshow(mask_pred_infilt, cmap=cmap_infilt, alpha=0.8) # Aplicando color sólido
        axes[2, 2].set_title('Máscara Final (Cascada > 0.5)', fontsize=14); axes[2, 2].axis('off')

        plt.tight_layout()
        plt.show()

    slider = widgets.IntSlider(min=0, max=max_z, step=1, value=corte_z_optimo, description='Corte Z:')
    widgets.interact(plot_campeon_analysis, corte_z=slider)
    
else:
    print(f"ID {ID_BUSCADO} no localizado.")

[VAL] Cargados 6 casos de UPenn-GBM (Pipeline 2)
[VAL] Cargados 21 casos de MU-Glioma Post (Pipeline 2)
Cargando Modelos en Cascada...


interactive(children=(IntSlider(value=69, description='Corte Z:', max=154), Output()), _dom_classes=('widget-i…